Pruned Pydantic model of the ARLEM specification of an AR education activity.
Has only what is needed for a single-person, mostly virtual, academic environment, linear experience.

In [ ]:
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field, HttpUrl


In [ ]:

# ==========================================
# PART 1: SIMPLIFIED WORKPLACE (Environment)
# ==========================================

class POI(BaseModel):
    """
    Points of Interest (offsets) relative to an anchor.
    """
    id: str = Field(..., description="Unique ID for this point.")
    x_offset: float = Field(0.0, description="X offset (cm).")
    y_offset: float = Field(0.0, description="Y offset (cm).")
    z_offset: float = Field(0.0, description="Z offset (cm).")
    # Rotations kept as they are essential for placing virtual content correctly
    x_rotation: float = Field(0.0, description="Pitch (degrees).")
    y_rotation: float = Field(0.0, description="Yaw (degrees).")
    z_rotation: float = Field(0.0, description="Roll (degrees).")

class Detectable(BaseModel):
    """
    Tracking configuration. For virtual labs, usually just one 'origin' anchor is needed.
    """
    id: str = Field(..., description="ID referenced by the workplace origin.")
    type: Literal["marker", "anchor"] = Field(..., description="Use 'marker' for QR codes, 'anchor' for spatial anchors.")
    url: Optional[HttpUrl] = Field(None, description="Asset URL if a specific marker image is required.")

class Place(BaseModel):
    """
    A location in the virtual space where content can appear.
    """
    id: str = Field(..., description="Unique ID for this location.")
    name: str = Field(..., description="Human-readable name (e.g., 'Experiment Zone').")
    detectable: Optional[str] = Field(None, description="ID of the anchor/marker this place is relative to.")
    pois: List[POI] = Field(default_factory=list, description="Specific offsets for content placement.")

class AugmentationPrimitive(BaseModel):
    """
    The media assets (3D models, Audio, Video, Labels).
    """
    id: str = Field(..., description="Unique ID for this asset.")
    type: Literal["animation", "image", "video", "audio", "label"] = Field(..., description="Media type.")
    url: Optional[HttpUrl] = Field(None, description="URL to the GLB/FBX model or media file.")
    scale: Optional[float] = Field(1.0, description="Scale factor.")

class Workplace(BaseModel):
    """
    A simplified environment definition for virtual/academic use.
    Removed: Persons, Sensors, Devices, Apps, WarningSigns.
    """
    id: str = Field(..., description="Unique ID for the lab environment.")
    name: str = Field(..., description="Name of the lab.")
    origin: str = Field(..., description="ID of the Detectable that acts as (0,0,0).")
    
    # We keep 'places' for locations and 'things' for virtual props if needed
    places: List[Place] = Field(default_factory=list, description="Locations relative to the origin.")
    
    detectables: List[Detectable] = Field(default_factory=list, description="Tracking definitions.")
    primitives: List[AugmentationPrimitive] = Field(default_factory=list, description="3D models and media assets.")
    predicates: List[AugmentationPrimitive] = Field(default_factory=list, description="Reusable verbs (e.g., 'Highlight', 'Rotate').")

# ==========================================
# PART 2: LINEAR ACTIVITY (Logic)
# ==========================================

class Instruction(BaseModel):
    """
    Educational content displayed to the student.
    """
    title: str = Field(..., description="Step title.")
    description: str = Field(..., description="Detailed instructions for the student.")

class Activate(BaseModel):
    """
    Command to show content.
    """
    target: str = Field(..., description="ID of the Place (location) or Action.")
    type: Literal["primitive", "predicate", "action"] = Field(..., description="Type of item to activate.")
    augmentation: Optional[str] = Field(None, description="ID of the Primitive/Predicate to show.")
    poi: Optional[str] = Field(None, description="Where to place it (POI ID).")
    state: Optional[str] = Field(None, description="Animation state/keyframe.")
    viewport: Optional[Literal["actions"]] = Field(None, description="Use 'actions' to attach to HUD.")

class Deactivate(BaseModel):
    """
    Command to hide content.
    """
    target: str = Field(..., description="Target ID or '*' for clear all.")
    augmentation: Optional[str] = Field(None, description="Specific item to remove.")

class Trigger(BaseModel):
    """
    Simplified Event Listener.
    Removed: Sensors, Voice (optional), logic operators.
    """
    mode: Literal["click", "detect"] = Field(..., description="User clicks 'Next' or looks at an object.")
    id: str = Field(..., description="ID of the entity to interact with.")
    duration: Optional[int] = Field(None, description="Time (ms) required for detection.")

class ActionFlow(BaseModel):
    """
    Simplified Flow.
    Removed: Messages, IfLogic (enforcing linearity).
    """
    remove_self: bool = Field(False, alias="removeSelf", description="Clear instruction upon completion?")
    activate: List[Activate] = Field(default_factory=list, description="Show these items.")
    deactivate: List[Deactivate] = Field(default_factory=list, description="Hide these items.")

class Action(BaseModel):
    """
    A single linear step.
    """
    id: str = Field(..., description="Step ID.")
    instruction: Optional[Instruction] = Field(None, description="Text for this step.")
    enter: Optional[ActionFlow] = Field(None, description="Execute on start.")
    exit: Optional[ActionFlow] = Field(None, description="Execute on completion.")
    triggers: List[Trigger] = Field(default_factory=list, description="Conditions to move to next step.")

class Activity(BaseModel):
    """
    Linear academic activity.
    """
    id: str = Field(..., description="Activity ID.")
    name: str = Field(..., description="Activity Title.")
    workplace: Union[HttpUrl, str] = Field(..., description="Ref to Workplace.")
    start: str = Field(..., description="First Action ID.")
    actions: List[Action] = Field(default_factory=list, description="Sequence of steps.")